# Cấp 8 – AlphaZero-inspired trên Google Colab

1. **Runtime → Change runtime type → T4 GPU.**
2. Chạy lần lượt các ô. Mọi checkpoint, replay buffer và log lưu trên **Google Drive** (`MyDrive/caro/alphazero2`).
3. **Bị ngắt (hết quota, mất kết nối) thì mở lại notebook và chạy lại từ đầu**: chương trình đọc `state.json` trên Drive và làm tiếp từ chỗ dừng.

Thư mục cũ `MyDrive/caro/alphazero` là lần chạy đầu bị lỗi (PUCT chưa có radius/FPU, mạng không học được): giữ lại làm minh chứng, **không chạy tiếp vào đó**.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%cd /content
!test -d TTNTNC-NHOM31 || git clone https://github.com/minhhieu310800-jpg/TTNTNC-NHOM31.git
%cd /content/TTNTNC-NHOM31
!git pull --ff-only
%cd do-an-caro
!pip install -q -r requirements.txt

/content
Cloning into 'TTNTNC-NHOM31'...
remote: Enumerating objects: 758, done.
remote: Counting objects: 100% (296/296), done.
remote: Compressing objects: 100% (161/161), done.
remote: Total 758 (delta 150), reused 181 (delta 134), pack-reused 462 (from 1)
Receiving objects: 100% (758/758), 11.65 MiB | 16.57 MiB/s, done.
Resolving deltas: 100% (324/324), done.
/content/TTNTNC-NHOM31
Already up to date.
/content/TTNTNC-NHOM31/do-an-caro


## Huấn luyện

12 vòng × 200 ván, 5 epoch/vòng. `--max-hours`: tự dừng an toàn trước khi hết quota GPU (dừng sau nhóm ván đang chơi). Tăng `--rounds` nếu muốn chạy thêm vòng.

In [3]:
OUT = '/content/drive/MyDrive/caro/alphazero2'
!python -m training.alphazero configs/alphazero.yaml --device cuda --out-dir {OUT} --rounds 25 --max-hours 3

[04:20:53] bắt đầu ở vòng 21 (selfplay), thiết bị cuda, thư mục /content/drive/MyDrive/caro/alphazero2
[04:21:34] vòng 21: self-play 20/200 ván
[04:22:01] vòng 21: self-play 40/200 ván
[04:22:33] vòng 21: self-play 60/200 ván
[04:23:01] vòng 21: self-play 80/200 ván
[04:23:30] vòng 21: self-play 100/200 ván
[04:24:00] vòng 21: self-play 120/200 ván
[04:24:31] vòng 21: self-play 140/200 ván
[04:25:00] vòng 21: self-play 160/200 ván
[04:25:34] vòng 21: self-play 180/200 ván
[04:26:04] vòng 21: self-play 200/200 ván
[04:26:53] vòng 21: train xong, policy_loss 2.184, value_loss 0.730
[04:30:04] vòng 21: gating score 0.62 -> NHẬN
[04:30:37] vòng 22: self-play 20/200 ván
[04:31:20] vòng 22: self-play 40/200 ván
[04:31:54] vòng 22: self-play 60/200 ván
[04:32:23] vòng 22: self-play 80/200 ván
[04:32:51] vòng 22: self-play 100/200 ván
[04:33:22] vòng 22: self-play 120/200 ván
[04:33:56] vòng 22: self-play 140/200 ván
[04:34:24] vòng 22: self-play 160/200 ván
[04:34:52] vòng 22: self-play 180/2

## Xem tiến độ

In [4]:
import json, pandas as pd
rows = [json.loads(l) for l in open(f'{OUT}/train_log.jsonl')]
pd.DataFrame(rows)[['round', 'policy_loss', 'value_loss', 'gate_score', 'accepted', 'best_round',
                   'selfplay_mean_len', 'selfplay_x_win', 'selfplay_s', 'gate_s']]

,round,policy_loss,value_loss,gate_score,accepted,best_round,selfplay_mean_len,selfplay_x_win,selfplay_s,gate_s
0,1,4.257419,0.730963,0.650,True,1,40.605,0.535,390.886732,325.268771
1,2,4.078516,0.638182,0.800,True,2,33.130,0.530,325.307571,283.589223
2,3,3.900360,0.578843,0.700,True,3,25.840,0.530,254.009688,225.146892
3,4,3.761876,0.538568,0.700,True,4,20.360,0.350,197.314282,213.867732
4,5,3.346654,0.602209,0.650,True,5,20.755,0.315,208.497639,200.113724
5,6,2.980784,0.611195,0.750,True,6,19.080,0.280,191.133411,203.232636
6,7,2.767878,0.616774,0.675,True,7,20.235,0.325,209.881523,231.471314
7,8,2.600718,0.616301,0.700,True,8,15.580,0.270,152.998998,157.712728
8,9,2.483039,0.629951,0.700,True,9,16.655,0.495,169.142344,161.274239
9,10,2.450346,0.654687,0.825,True,10,20.765,0.655,227.522366,144.052338
